# **Assignment 9: Neural Networks**

## Author: Mohammad Mehdi Zare, Mohammad Ali Olama

#Load Libraries

In [1]:
import pandas as pd
import io
import numpy as np
from sklearn.preprocessing import StandardScaler
import sklearn
import os
import copy
import seaborn as sns #visualisation
import matplotlib.pyplot as plt #visualisation
%matplotlib inline
sns.set(color_codes=True)

In [2]:
# Diable warnings about max_itr not reached etc.!
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

#Upload the Datasets
For this assignment, we used two different datasets: breast cancer dataset, which we will use to do the classification part, and the bike sharing dataset, which we will use to do the regression part.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Regression dataset

In [4]:
data = pd.read_csv("/content/drive/MyDrive/TEIAS/ADS/HW9/day.csv")
data.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


##Preprocessing

In [5]:
# Convert the 'dteday' column to datetime
data['dteday'] = pd.to_datetime(data['dteday'])

# Extract day, month, and year into separate columns
data['day'] = data['dteday'].dt.day
data['month'] = data['dteday'].dt.month
data['year'] = data['dteday'].dt.year

# Optionally, you can drop the original 'dteday' column if you don't need it anymore
data.drop('dteday', axis=1, inplace=True)

# Display the updated DataFrame
data.head()

,instant,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,day,month,year
0,1,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985,1,1,2011
1,2,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801,2,1,2011
2,3,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349,3,1,2011
3,4,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562,4,1,2011
4,5,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600,5,1,2011


In [6]:
# Drop the 'instant' column
data.drop('instant', axis=1, inplace=True)

# Display the updated DataFrame
data.head()

,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,day,month,year
0,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985,1,1,2011
1,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801,2,1,2011
2,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349,3,1,2011
3,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562,4,1,2011
4,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600,5,1,2011


In [7]:
# Define the target variable 'y' as the 'cnt' column
y = data['cnt']

# Define the features 'X' by dropping the target column and any other columns you don't need
X = data.drop(columns=['cnt'])

# Display the shapes of X and y to confirm
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (731, 16)
Target shape: (731,)


In [8]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
bike_X_train, bike_X_test, bike_y_train, bike_y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##Classification dataset

In [9]:
# os.environ['KAGGLE_CONFIG_DIR'] = "/content/gdrive/My Drive/Kaggle"
# ! kaggle datasets download -d yasserh/breast-cancer-dataset
# !unzip \*.zip && rm *.zip
df = pd.read_csv('breast-cancer.csv')
print(df.shape)

(569, 32)


In [10]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


##Preprocessing

In [11]:
null_columns = []
for col_name in list(df.columns):
  no_null_values = sum(df[col_name].isnull())
  if no_null_values:
    print('Number of null values in column %s is %d' %(col_name,no_null_values))
    null_columns.append(col_name)

In [12]:
df.dropna(inplace=True)
df.shape

(569, 32)

In [13]:
df.drop(['id'],axis=1,inplace=True)

In [14]:
# First, let's separate the target column, price
target_variable = df.pop('diagnosis')

In [15]:
from sklearn.preprocessing import OneHotEncoder,LabelBinarizer
enc = LabelBinarizer()
y = enc.fit_transform(target_variable)

#Split dataset

In [16]:
from sklearn.model_selection import train_test_split, cross_val_score

cancer_X_train, cancer_X_test, cancer_y_train, cancer_y_test = train_test_split(df,y,test_size=0.2,random_state=42)

print(cancer_X_train.shape)
print(cancer_X_test.shape)

(455, 30)
(114, 30)


#Classification

The code of multi-layer perceptron is straightforward. We first initialize a class of type `MLPClassifier`, which has many parameters, including solver, the algorithm to use to solve the optimization problem. The `lbfgs` solver is an optimizer in the family of quasi-Newton methods. `hidden_layer_size` parameter refers to the shape of our network. After that, we will fit the classifier on our data and evaluate it on the test set.

In [17]:
from sklearn.neural_network import MLPClassifier
clf = MLPClassifier(hidden_layer_sizes=(30,16,8,4 ,2), max_iter=1000, random_state=42)
clf.fit(cancer_X_train, cancer_y_train)

MLPClassifier(hidden_layer_sizes=(30, 16, 8, 4, 2), max_iter=1000,
              random_state=42)

In [18]:
from sklearn.metrics import f1_score
f1_score(cancer_y_test, clf.predict(cancer_X_test))

0.9347826086956522

##Keras

Using Tensorflow sequential API makes it easy to create different types of networks. We first create the model using instantiating the class of `tf.models.Sequential`, and after that, we will add the layers and corresponding activation function to it. Note that we use the ReLU activation function for internal layers, while we use the sigmoid function for the last layer as this is a classification task, and we need the last logit value to be between 0 and 1. Finally, we will fit the model on our training data and evaluate it on the test set using our custom f1-score function.

In [19]:
from keras.models import Sequential
from keras.layers import Dense
model = Sequential()
model.add(Dense(21, activation='relu'))
model.add(Dense(15, activation='relu'))
model.add(Dense(10, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [20]:
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(cancer_X_train, cancer_y_train, epochs=50, batch_size=32, validation_split=0.2)


# Predict probabilities on the test set
y_pred_prob = model.predict(cancer_X_test)

# Convert probabilities to class labels (0 or 1)
y_pred = (y_pred_prob > 0.5).astype(int)

# Calculate the F1 score
f1 = f1_score(cancer_y_test, y_pred)
print(f"F1 Score: {f1}")

Epoch 1/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.5917 - loss: 9.1568 - val_accuracy: 0.3626 - val_loss: 3.1381
Epoch 2/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4085 - loss: 2.3860 - val_accuracy: 0.6813 - val_loss: 0.8204
Epoch 3/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7324 - loss: 0.6843 - val_accuracy: 0.9231 - val_loss: 0.3046
Epoch 4/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8816 - loss: 0.2833 - val_accuracy: 0.9011 - val_loss: 0.2330
Epoch 5/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9237 - loss: 0.2506 - val_accuracy: 0.9231 - val_loss: 0.2488
Epoch 6/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9275 - loss: 0.2195 - val_accuracy: 0.8901 - val_loss: 0.2263
Epoch 7/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9245 - loss: 0.2136 - val_accuracy: 0.8901 - val_loss: 0.2240
Epoch 8/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9462 - loss: 0.1760 - val_accuracy: 0.9121 - val_

##Torch
Using the PyTorch library is quite similar to Tensorflow. The PyTorch library has a similar API to Tensorflow's `Sequential` API, but in this part, we will use the standard PyTorch API. First, we will transform the training and test data arrays to PyTorch datasets and dataloaders. After that, we create a model using PyTorch API. In the `__init__` method of this class, we will initialize different linear layers of the model. The difference between PyTorch and Tensorflow is that in PyTorch, we need to know the shape of our input layer, while in Tensorflow, the library can find that itself. In the `forward` method, we will specify how the model's components interact with each other. The other difference between PyTorch and Tensorflow is that we should write the training loop ourselves in PyTorch, while in Tensorflow, the framework will take this part over. Finally, like other parts, we will evaluate the model on the test set.

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42)

In [22]:
import torch
from torch.utils.data import Dataset, DataLoader
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
if (device.type == 'cuda'):
    print(torch.cuda.get_device_properties(0))
    print(torch.cuda.empty_cache())
    print(torch.cuda.mem_get_info())

class BikeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).reshape(-1,1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

cpu


In [23]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.fc1 = nn.Linear(30, 15)
        self.fc2 = nn.Linear(15, 10)
        self.fc3 = nn.Linear(10, 5)
        self.fc4 = nn.Linear(5, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = torch.sigmoid(self.fc4(x))
        return x

model = Model()
print(model)

Model(
  (fc1): Linear(in_features=30, out_features=15, bias=True)
  (fc2): Linear(in_features=15, out_features=10, bias=True)
  (fc3): Linear(in_features=10, out_features=5, bias=True)
  (fc4): Linear(in_features=5, out_features=1, bias=True)
)


In [24]:
train_dataset = BikeDataset(cancer_X_train, cancer_y_train)
test_dataset = BikeDataset(cancer_X_test, cancer_y_test)

train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)


In [25]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
num_epochs = 50

def train():
    model.train()
    tot_loss = 0.0
    N = 0
    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)
        N += labels.shape[0]

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        tot_loss += loss
        loss.backward()
        optimizer.step()
    print(tot_loss / N)


for epoch in range(num_epochs):
    train()


tensor(0.0072, grad_fn=<DivBackward0>)
tensor(0.0067, grad_fn=<DivBackward0>)
tensor(0.0066, grad_fn=<DivBackward0>)
tensor(0.0064, grad_fn=<DivBackward0>)
tensor(0.0064, grad_fn=<DivBackward0>)
tensor(0.0063, grad_fn=<DivBackward0>)
tensor(0.0062, grad_fn=<DivBackward0>)
tensor(0.0060, grad_fn=<DivBackward0>)
tensor(0.0057, grad_fn=<DivBackward0>)
tensor(0.0055, grad_fn=<DivBackward0>)
tensor(0.0049, grad_fn=<DivBackward0>)
tensor(0.0047, grad_fn=<DivBackward0>)
tensor(0.0042, grad_fn=<DivBackward0>)
tensor(0.0037, grad_fn=<DivBackward0>)
tensor(0.0028, grad_fn=<DivBackward0>)
tensor(0.0026, grad_fn=<DivBackward0>)
tensor(0.0023, grad_fn=<DivBackward0>)
tensor(0.0031, grad_fn=<DivBackward0>)
tensor(0.0039, grad_fn=<DivBackward0>)
tensor(0.0033, grad_fn=<DivBackward0>)
tensor(0.0028, grad_fn=<DivBackward0>)
tensor(0.0027, grad_fn=<DivBackward0>)
tensor(0.0025, grad_fn=<DivBackward0>)
tensor(0.0023, grad_fn=<DivBackward0>)
tensor(0.0023, grad_fn=<DivBackward0>)
tensor(0.0023, grad_fn=<D

In [26]:
from sklearn.metrics import f1_score

true_labels =[]
predicted_labels = []

with torch.no_grad():
    model.eval()
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        predicted = torch.round(outputs.flatten())

        criterion(outputs, labels)

        true_labels.extend(labels.cpu().numpy())
        predicted_labels.extend(predicted.cpu().numpy())
    print(f1_score(true_labels, predicted_labels))

0.963855421686747


##Non-Sequential Keras
The non-sequential API of the Tensorflow is much more similar to the PyTorch standard API. This section will do a similar procedure to the previous section using non-sequential API.

In [27]:
from keras.layers import Add, Concatenate, Dense, Input
from keras.models import Model
import tensorflow as tf

In [28]:
# this block is adopted from the class lab.
from keras.layers import Add, Concatenate
from keras.models import Model

input_layer = Input(shape=(cancer_X_train.shape[1],))

# Pass the output of the input layer into a hidden layer
hidden_layer_1 = Dense(20,activation='relu')(input_layer)

# The second hidden layer receives input from the input_layer and not the last hidden layer
hidden_layer_2 = Dense(10,activation='relu')(input_layer)

# The second hidden layer receives input from the input_layer and not the last hidden layer
hidden_layer_3 = Dense(10,activation='relu')(hidden_layer_1)


#added = tf.keras.layers.Add()([hidden_layer_1, hidden_layer_2])
#add_layer = Add()([hidden_layer_2,hidden_layer_1])

add_layer = Concatenate()([hidden_layer_3, hidden_layer_2])

output_layer = Dense(1,activation='sigmoid')(add_layer)

# Defined an actual model out of the above with an output being the same shape as the above layer and input being the size as the input layer
model = Model(inputs=input_layer, outputs=output_layer)

In [29]:
model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    metrics=[tf.keras.metrics.Precision(),tf.keras.metrics.Recall()],
    )
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1             │ (None, 30)             │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_4 (Dense)           │ (None, 20)             │            620 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_6 (Dense)           │ (None, 10)             │            210 │ dense_4[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_5 (Dense)           │ (None, 10)             │            310 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concatenate) │ (None, 20)             │              0 │ dense_6[0][0],         │
│                           │                        │                │ dense_5[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_7 (Dense)           │ (None, 1)              │             21 │ concatenate[0][0]      │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 1,161 (4.54 KB)

 Trainable params: 1,161 (4.54 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history = model.fit(cancer_X_train, cancer_y_train,
                    batch_size = 256,
                    epochs = 50,
                    validation_split=0.2,
                    )

Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 382ms/step - loss: 106.6880 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 76.9139 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 73.4982 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 39.5357 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - loss: 32.6941 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_loss: 5.2734 - val_precision: 0.0755 - val_recall: 0.1212
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 6.8095 - precision: 0.1988 - recall: 0.3571 - val_loss: 20.6967 - val_precision: 0.3626 - val_recall: 1.0000
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 20.9140 - precision: 0.3754 - recall: 1.0000 - val_loss: 27.3249 - val_precision: 0.3626 - val_recall: 1.0000
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 26.4710 - precision: 0.3780 - recall: 1.0000 - val_loss: 25.9302 - v

In [31]:
predictions = model.predict(cancer_X_test)
print('Macro F1-score is %.9f' %f1_score(cancer_y_test, predictions>0.5, average='macro'))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
Macro F1-score is 0.934348005


#Regression

##ScikitLearn
Using the ScikitLearn library to do the regression is quite similar to classifying using this library.

###Model

In [32]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score

# Initialize the MLPRegressor
mlp_regressor = MLPRegressor(hidden_layer_sizes=(64, 32), activation='relu', solver='adam', max_iter=500, random_state=42)

# Train the model
mlp_regressor.fit(bike_X_train, bike_y_train)

# Make predictions on the test set
bike_y_pred = mlp_regressor.predict(bike_X_test)

# Evaluate the model using R-squared
r2 = r2_score(bike_y_test, bike_y_pred)
print(f"R-squared score: {r2}")

R-squared score: 0.999946177675794


##Keras

###Model

In [33]:
from keras.models import Sequential
from keras.layers import Dense
from keras import backend as K
from sklearn.metrics import r2_score


keras.utils.set_random_seed(42)

# Define the model
model = Sequential()
model.add(Dense(21, activation='relu', input_shape=(bike_X_train.shape[1],)))  # Input layer
model.add(Dense(10, activation='relu'))  # Hidden layer 1
model.add(Dense(5, activation='relu'))  # Hidden layer 2
model.add(Dense(2, activation='relu')) # Hidden layer 3
model.add(Dense(1))  # Output layer (no activation for regression)

# Compile the model
model.compile(optimizer='adam', loss='mse')

# Train the model
model.fit(bike_X_train, bike_y_train, epochs=50, batch_size=128, verbose=1) # set verbose = 1 to see training progress

# Make predictions on the test set
bike_y_pred = model.predict(bike_X_test)

# Evaluate the model
r2 = r2_score(bike_y_test, bike_y_pred)
print(f"R-squared score: {r2}")

Epoch 1/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 25722386.0000
Epoch 2/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 25381602.0000 
Epoch 3/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 25087496.0000 
Epoch 4/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 24837332.0000 
Epoch 5/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 24622600.0000 
Epoch 6/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 24349928.0000 
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 23791332.0000 
Epoch 8/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 23203580.0000 
Epoch 9/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 22911528.0000 
Epoch 10/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 22510324.0000 
Epoch 11/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 22196316.0000 
Epoch 12/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 21782420.0000 
Epoch 13/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 21344316.0000 
Epoch 14/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 20831590.0000 
Ep

1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
R-squared score: 0.9796738624572754


## Torch

In [34]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(35)

###Model Initialization

In [35]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.fc1 = nn.Linear(16, 25)
        self.fc2 = nn.Linear(25, 10)
        self.fc3 = nn.Linear(10, 5)
        self.fc4 = nn.Linear(5, 2)
        self.fc5 = nn.Linear(2, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = self.fc5(x)
        return x

model = Model()
print(model)

Model(
  (fc1): Linear(in_features=16, out_features=25, bias=True)
  (fc2): Linear(in_features=25, out_features=10, bias=True)
  (fc3): Linear(in_features=10, out_features=5, bias=True)
  (fc4): Linear(in_features=5, out_features=2, bias=True)
  (fc5): Linear(in_features=2, out_features=1, bias=True)
)


In [36]:
train_dataset = BikeDataset(bike_X_train, np.array(bike_y_train))
test_dataset = BikeDataset(bike_X_test, np.array(bike_y_test))

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

###Model Training

In [37]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50

def train():
    model.train()
    tot_loss = 0.0
    N = 0
    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)
        N += labels.shape[0]

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        tot_loss += loss
        loss.backward()
        optimizer.step()
    print(tot_loss / N)


for epoch in range(num_epochs):
    train()


tensor(802972.6875, grad_fn=<DivBackward0>)
tensor(786720.2500, grad_fn=<DivBackward0>)
tensor(798825.1250, grad_fn=<DivBackward0>)
tensor(762388.9375, grad_fn=<DivBackward0>)
tensor(722091.0625, grad_fn=<DivBackward0>)
tensor(648961.4375, grad_fn=<DivBackward0>)
tensor(494367.0625, grad_fn=<DivBackward0>)
tensor(270434.7188, grad_fn=<DivBackward0>)
tensor(75638.4531, grad_fn=<DivBackward0>)
tensor(16296.3496, grad_fn=<DivBackward0>)
tensor(13943.3545, grad_fn=<DivBackward0>)
tensor(13112.4824, grad_fn=<DivBackward0>)
tensor(12755.0400, grad_fn=<DivBackward0>)
tensor(11599.9434, grad_fn=<DivBackward0>)
tensor(10902.7021, grad_fn=<DivBackward0>)
tensor(9814.7471, grad_fn=<DivBackward0>)
tensor(9133.3057, grad_fn=<DivBackward0>)
tensor(7993.3203, grad_fn=<DivBackward0>)
tensor(6813.6465, grad_fn=<DivBackward0>)
tensor(5845.4282, grad_fn=<DivBackward0>)
tensor(4788.6299, grad_fn=<DivBackward0>)
tensor(3606.0925, grad_fn=<DivBackward0>)
tensor(2601.1011, grad_fn=<DivBackward0>)
tensor(1808

###Model Evaluation

In [38]:
from sklearn.metrics import r2_score

true_labels =[]
predicted_labels = []

with torch.no_grad():
    model.eval()
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)

        criterion(outputs, labels)

        true_labels.extend(labels.cpu().numpy())
        predicted_labels.extend(outputs.cpu().numpy())
    print(r2_score(true_labels, predicted_labels))

0.9998818090681943


##Non-Sequential Keras

In [39]:
from keras.layers import Add, Concatenate, Dense, Input
from keras.models import Model
import tensorflow as tf

###Model Initialization

In [40]:
input_layer = Input(shape=(bike_X_train.shape[1],))
hidden_layer_1 = Dense(64, activation='relu')(input_layer)
hidden_layer_2 = Dense(32, activation='relu')(hidden_layer_1)
hidden_layer_3 = Dense(16, activation='relu')(hidden_layer_2)
output_layer = Dense(1, activation=None)(hidden_layer_3)

model = Model(inputs=input_layer, outputs=output_layer)

In [41]:
from keras.layers import Add, Concatenate
from keras.models import Model

input_layer = Input(shape=(bike_X_train.shape[1],))

# Pass the output of the input layer into a hidden layer
hidden_layer_1 = Dense(20,activation='relu')(input_layer)

# The second hidden layer receives input from the input_layer and not the last hidden layer
hidden_layer_2 = Dense(10,activation='relu')(input_layer)

# The second hidden layer receives input from the input_layer and not the last hidden layer
hidden_layer_3 = Dense(10,activation='relu')(hidden_layer_1)


#added = tf.keras.layers.Add()([hidden_layer_1, hidden_layer_2])
#add_layer = Add()([hidden_layer_2,hidden_layer_1])

add_layer = Concatenate()([hidden_layer_3, hidden_layer_2])

output_layer = Dense(1,activation=None)(add_layer)

# Defined an actual model out of the above with an output being the same shape as the above layer and input being the size as the input layer
model = Model(inputs=input_layer, outputs=output_layer)

In [42]:
model.compile(
    loss='mse',
    optimizer=tf.keras.optimizers.Adam(),
    )
model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4             │ (None, 16)             │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_17 (Dense)          │ (None, 20)             │            340 │ input_layer_4[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_19 (Dense)          │ (None, 10)             │            210 │ dense_17[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_18 (Dense)          │ (None, 10)             │            170 │ input_layer_4[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_1             │ (None, 20)             │              0 │ dense_19[0][0],        │
│ (Concatenate)             │                        │                │ dense_18[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_20 (Dense)          │ (None, 1)              │             21 │ concatenate_1[0][0]    │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 741 (2.89 KB)

 Trainable params: 741 (2.89 KB)

 Non-trainable params: 0 (0.00 B)

###Model Training

In [43]:
history = model.fit(bike_X_train, bike_y_train,
                    batch_size = 16,
                    epochs = 50)

Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 26072378.0000
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 14990822.0000
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 7949792.5000
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3492296.0000
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1420880.6250
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 606121.3750
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 373764.6875 
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 320399.3438 
Epoch 9/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 300326.3750 
Epoch 10/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 283247.3750
Epoch 11/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 265901.8438
Epoch 12/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 247346.8906 
Epoch 13/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 220404.6562
Epoch 14/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 190922.3438 
Epo

# **Assignment 8: Multi-class Classification**

## Author: Mohammad Mehdi Zare, Mohammad Ali Olama

###Model Evaluation

In [44]:
r2_score(bike_y_test, model.predict(bike_X_test))


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


0.9998752474784851

### Explain why neural networks are so powerful and what the diffcult part is in designing neural networks.

Neural networks are powerful because they can model complex, non-linear relationships in data through layers of interconnected nodes (neurons). This allows them to solve tasks like image recognition and language processing that are difficult for traditional models. Their ability to learn features directly from raw data eliminates the need for manual feature engineering, and their scalability allows them to improve as they are exposed to more data.

The main difficulty in designing neural networks lies in choosing the right architecture, such as the number of layers, types of activation functions, and regularization methods. Training a neural network requires careful tuning of hyperparameters (like learning rate), avoiding overfitting, and managing computational resources efficiently. Balancing these factors is key to achieving optimal performance.